# Kaggle Submission — LSTM Model
**F20AA CW2 — Google Maps Review Challenge**

Generates Kaggle-ready CSV using LSTM from Task 5, trained on the full dataset with improved parameters.
- vocab_size: 30,000
- max_length: 200
- epochs: up to 10 with early stopping

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import os

from task5_torch_helpers import (
    DEVICE,
    Dense,
    Dropout,
    EarlyStopping,
    Embedding,
    LSTM,
    Sequential,
    Tokenizer,
    pad_sequences,
)

print(f'Using backend on: {DEVICE}')
print('Imports done.')


Using backend on: cpu
Imports done.


## 2. Load Full Training Data

In [2]:
print('Loading full training data...')
df_train = pd.read_csv('data/train_english.csv')
print(f'Training samples: {len(df_train):,}')
df_train.head(2)


Loading full training data...
Training samples: 271,897


,text,rating,text_length
0,This place is TERRIBLE; the people in charge a...,2,551
1,Terrible Service! And they are saying that I n...,1,258


## 3. Tokenise & Pad Full Training Data

Fitting tokenizer on all 271,897 reviews for better vocabulary coverage on the Kaggle test set.

In [3]:
# Improved parameters vs Task 5
VOCAB_SIZE  = 30000   # increased from 10,000
MAX_LENGTH  = 200     # increased from 100
OOV_TOKEN   = '<OOV>'

X_train_text = df_train['text'].astype(str)
y_train      = (df_train['rating'] - 1).values

print(f'Training samples: {len(X_train_text):,}')
print(f'Label range: {y_train.min()} - {y_train.max()}')


Training samples: 271,897
Label range: 0 - 4


In [4]:
print('Fitting tokenizer on full training data...')
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train_text)

print('Tokenizing and padding...')
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')

# 5% split for early stopping only
from sklearn.model_selection import train_test_split as _split
X_fit, X_es, y_fit, y_es = _split(
    X_train_pad, y_train, test_size=0.05, random_state=42, stratify=y_train
)

print(f'Training matrix shape: {X_train_pad.shape}')
print(f'Fit: {X_fit.shape}, Early stop: {X_es.shape}')


Fitting tokenizer on full training data...
Tokenizing and padding...
Training matrix shape: (271897, 200)
Fit: (258302, 200), Early stop: (13595, 200)


## 4. Build & Train LSTM

Same architecture as Task 5 but trained on full data with larger vocab and sequence length.

In [5]:
lstm_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LENGTH),
    LSTM(64),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(5, activation='softmax')
])

lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()


_TextClassifierModule(
  (embedding): Embedding(30000, 128, padding_idx=0)
  (recurrent): LSTM(128, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (hidden): Linear(in_features=64, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=5, bias=True)
)


In [6]:
print('Training LSTM...')

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_fit,
    y_fit,
    validation_data=(X_es, y_es),
    epochs=10,       # increased from 5
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

print('Training complete.')


Training LSTM...
Epoch 1/10 - loss: 1.3536 - accuracy: 0.4613 - val_loss: 1.1441 - val_accuracy: 0.5834
Epoch 2/10 - loss: 1.0403 - accuracy: 0.5963 - val_loss: 0.9201 - val_accuracy: 0.6255
Epoch 3/10 - loss: 0.9028 - accuracy: 0.6317 - val_loss: 0.8844 - val_accuracy: 0.6429
Epoch 4/10 - loss: 0.8360 - accuracy: 0.6642 - val_loss: 0.8456 - val_accuracy: 0.6630
Epoch 5/10 - loss: 0.7925 - accuracy: 0.6823 - val_loss: 0.8349 - val_accuracy: 0.6656
Epoch 6/10 - loss: 0.7594 - accuracy: 0.6958 - val_loss: 0.8404 - val_accuracy: 0.6585
Epoch 7/10 - loss: 0.7283 - accuracy: 0.7105 - val_loss: 0.8615 - val_accuracy: 0.6536
Epoch 8/10 - loss: 0.6984 - accuracy: 0.7242 - val_loss: 0.8561 - val_accuracy: 0.6594
Training complete.


## 5. Load Kaggle Test Data & Generate Predictions

In [7]:
print('Loading Kaggle test data...')
test = pd.read_csv('data/kaggle_test.csv')
test.columns = [c.strip().lower() for c in test.columns]
print(f'Test shape: {test.shape}')
test.head()


Loading Kaggle test data...
Test shape: (89100, 2)


,id,text
0,1,Quite easy to rent a car bur it is not easy to...
1,2,Nice voleyball court close to restaurants and ...
2,3,"Very nice built homes, the future locations ar..."
3,4,This dental clinic appears to be friendly with...
4,5,We came in to discuss tattoos. Only person th...


In [8]:
print('Tokenizing and padding test sequences...')
X_test_seq = tokenizer.texts_to_sequences(test['text'].astype(str))
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
print(f'Test matrix shape: {X_test_pad.shape}')


Tokenizing and padding test sequences...
Test matrix shape: (89100, 200)


In [9]:
print('Generating predictions...')
pred_probs   = lstm_model.predict(X_test_pad, batch_size=256)
pred_classes = np.argmax(pred_probs, axis=1)  # 0-4
predictions  = pred_classes + 1               # convert back to 1-5

print(f'Predictions shape: {predictions.shape}')
print('\nPrediction distribution:')
print(pd.Series(predictions).value_counts().sort_index())


Generating predictions...
Predictions shape: (89100,)

Prediction distribution:
1    20797
2     3450
3    13752
4    45904
5     5197
Name: count, dtype: int64


## 6. Save Submission CSV

In [10]:
id_col = 'id' if 'id' in test.columns else test.columns[0]

submission = pd.DataFrame({
    'Id':     test[id_col].values,
    'Rating': predictions
})

assert submission.shape[1] == 2
assert list(submission.columns) == ['Id', 'Rating']
assert submission['Rating'].between(1, 5).all()
assert submission['Id'].is_unique
assert not submission.isnull().any().any()

print(f'All checks passed!')
print(f'Rows: {len(submission):,}')
submission.head(10)


All checks passed!
Rows: 89,100


,Id,Rating
0,1,4
1,2,4
2,3,4
3,4,1
4,5,5
5,6,4
6,7,4
7,8,5
8,9,1
9,10,3


In [11]:
os.makedirs('data/kaggle_submission', exist_ok=True)
output_path = 'data/kaggle_submission/submission_LSTM.csv'
submission.to_csv(output_path, index=False)
print(f'Saved to: {output_path}')

verify = pd.read_csv(output_path)
print(f'Verified rows: {len(verify):,}')
print(verify.head())


Saved to: data/kaggle_submission/submission_LSTM.csv
Verified rows: 89,100
   Id  Rating
0   1       4
1   2       4
2   3       4
3   4       1
4   5       5
